# 창원국가산단 ELECTRE TRI-B 재검증 (Phase 0-3)

이 노트북은 계산을 새로 하지 않는다. 모든 값은 `src/model/revalidation.py`와
`src/model/revalidation_phase3.py`가 만든 `outputs/tables/*.csv`를 읽어 표시만 한다.
판정 로직은 이 노트북에 없다.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from model import config  # noqa: E402

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
TABLES = ROOT / 'outputs/tables'
FIGS = ROOT / 'outputs/figures'

## (1) 재현 앵커 대조

`scripts/phase0_baseline.py`가 기록한 대조 결과를 그대로 불러온다.

In [ ]:
import json
hashes = json.loads((ROOT / 'outputs/revalidation_baseline_hashes.json').read_text(encoding='utf-8'))
print('동결 대상 소스 파일 수:', len(hashes.get('src_model_hashes', {})))
print('동결 대상 기존 테스트 파일 수:', len(hashes.get('tests_original_11_hashes', {})))
print(hashes.get('src_model_hashes_note'))

**해석**: Phase 0에서 확인한 결정적 기준값(v1.0 86/41/33/20, 후보 D 79/46/35/20, 강제 관찰 71행,
crisp 불일치 고유조합 1개)은 전부 재현되었고, 이후 모든 Phase는 그 위에서 진행됐다.

## (2) 비관/낙관 일치율 — 시나리오별

In [ ]:
opt = pd.read_csv(TABLES / 'electre_optimistic_vs_pessimistic.csv')
scorable = opt['pessimistic_stage'] != config.UNDETERMINED
summary = (opt[scorable].groupby(['candidate_id', 'scenario_id'])
           .agg(n_scorable=('agree', 'size'), n_agree=('agree', 'sum')).reset_index())
summary['agreement_rate'] = summary['n_agree'] / summary['n_scorable']
summary

**해석**: 기준·지속성중시 시나리오는 두 모형 모두 완전히 일치한다. 고용중시 시나리오(λ=0.60)만
v1.0 0.800 / D 0.913으로 어긋나며, 이는 배정절차(비관적 vs 낙관적)라는 임의 선택이 결과 일부를
좌우함을 뜻한다 — 어느 모형이 맞는지의 문제가 아니라 이 시나리오 고유의 특성이다.

## (3) 개정폭 교란 유지율

In [ ]:
pd.read_csv(TABLES / 'electre_input_perturbation_stability.csv')[
    ['candidate_id', 'scenario_id', 'retention_mean', 'retention_p05', 'retention_min',
     'zero_perturbation_self_check_mean']]

**해석**: 승수를 전부 1.0으로 두는 무교란 자가검증은 모든 조합에서 정확히 1.0이다(구조적 자기재현
확인). 실측 개정폭으로 자료를 교란하면 유지율이 0.90~0.93 범위로 낮아진다 — 이는 합격선
(0.95/0.90)에 못 미치는 수준이다.

## (4) 필연/가능 배정과 CAI — 공간 3종 비교

In [ ]:
display(pd.read_csv(TABLES / 'electre_robust_assignment_summary.csv')[
    ['space_id', 'n_samples', 'necessary_share_discriminating', 'necessary_share_scorable',
     'n_possible_1', 'n_possible_2', 'n_possible_3', 'used_for_acceptance', 'threshold', 'pass']])
display(Image(filename=str(FIGS / 'revalidation_subspace_comparison.png')))

**해석**: 참조사례와 양립하는 부분공간에서는 필연배정 비율이 0.011에서 0.101로 오르지만,
합격선 0.30에는 세 공간 모두 미치지 못한다. λ와 가중치의 허용 범위 자체가 넓기 때문이며,
AM1(p_upper 축소)은 가능단계 수 분포를 좁히는 데는 실제로 기여했지만 이 비율 자체를 바꾸지는
않았다.

## (5) 참조사례 양립성

In [ ]:
rcc = pd.read_csv(TABLES / 'electre_reference_case_compatibility.csv')
display(rcc[rcc.block_type == 'per_case'][['case_id', 'industry', 'quarter', 'relation', 'target_stage',
                                           'sample_pass_rate']])
display(rcc[rcc.block_type == 'fixed_model'][['model_id', 'scenario_id', 'rc1_pass', 'rc2_pass', 'rc3_pass',
                                              'all_pass', 'observed_stages']])

**해석**: 기준 시나리오에서 v1.0은 세 참조사례 중 둘(RC1·RC2)을 만족하지 못하고, D는 셋 다
만족한다 — 사전에 고정한 결정론적 대조 그대로다. 고용중시 시나리오에서는 v1.0도 양립하는데,
이는 가중치·λ 선택에 따라 달라질 수 있는 정상적인 결과다.

## (6) 동시점 독립지표 정합

In [ ]:
cev = pd.read_csv(TABLES / 'electre_concurrent_external_validation.csv')
cev[['model_id', 'scenario_id', 'indicator_column', 'role', 'spearman', 'discrimination_status',
     'sign_matches_expectation']]

**해석**: PPI 조정 생산 지표(보조 역할)는 두 모형 모두 기대한 부호와 일치하는 단조 관계를 보인다.
가동률·가동업체수(참고 역할)는 단계 판정에 쓰지 않으며, 가동업체수는 단계 간 차이가 없고
가동률은 단조적이지 않다.

## (7) 경계 프로파일 발견

In [ ]:
pd.read_csv(TABLES / 'electre_boundary_profile_findings.csv')

**해석**: g3의 명목-PPI조정 격차 75분위(약 6.80%p)가 경계 간격보다 커서 p 상한을 5.0으로
절단했다(BP1). g1은 일부 업종이 경계를 한 번도 통과하지 못했다(BP2). 강제 관찰 71행 중 7행은
g3만으로는 경계를 넘을 만큼 명목 생산이 감소했다(BP3) — 다음 개정 대상 후보다.

## (8) 재판정 표

In [ ]:
decision = pd.read_csv(TABLES / 'electre_revalidation_decision.csv')
decision[['model_id', 'scenario_id', 'c1_pass', 'c2_pass', 'c3_pass', 'c4_pass', 's1_pass',
         'overall_status', 'blocked_by']]

**해석**: 6행 모두 두 개 이상의 기준을 만족하지 못해 `BLOCKED_MULTIPLE`이다. S1(파라미터 공간
수준)과 C3(교란 유지율)는 모든 행에서 공통으로 미달이며, C2·C4는 시나리오에 따라 갈린다.
이 표는 채택 여부를 말하지 않는다 — 사전에 고정한 기준으로 각 항목이 통과했는지만 기록한다.

## (9) 2026Q2 CAI 표와 그림

In [ ]:
cai = pd.read_csv(TABLES / 'electre_class_acceptability_index_compatible.csv')
display(cai[cai.quarter == '2026Q2'].sort_values('industry')[
    ['industry', 'cai_observe', 'cai_check', 'cai_priority', 'cai_undetermined', 'necessary_stage',
     'possible_stages', 'v1_0_stage', 'd_stage']])
display(Image(filename=str(FIGS / 'revalidation_cai_compatible_latest.png')))

**해석**: 참조사례와 양립하는 부분공간으로 좁혀도 2026Q2 시점에서 여러 업종의 등급수용지수가
한 단계에 집중되지 않는다. 이 표는 특정 업종에 관한 결론이 아니라, 현재 파라미터 불확실성
아래에서 배정이 얼마나 갈릴 수 있는지를 보여주는 진단 자료다.